# Análise Avançada de TTS — Avaliação Comparativa de Modelos Open Source em PT-BR

Este notebook realiza uma análise aprofundada dos resultados do benchmark de Text-to-Speech (TTS),
expandindo as métricas originais com cálculos derivados, testes estatísticos rigorosos e
visualizações avançadas.

**Modelos avaliados:** Piper (VITS), Kokoro (StyleTTS2), XTTS v2 (GPT+HiFiGAN)  
**Dados de entrada:** `results/resultados_tts_tcc.csv` (420 sínteses · 3 modelos · 14 categorias)  
**Ambiente:** Google Colab (CPU-only) ou ambiente local compatível

---


## Seção 1 — Carregamento e Preparação dos Dados

Importação das bibliotecas necessárias e carregamento do arquivo de resultados brutos do benchmark.


In [ ]:
# Instalação de dependências adicionais (necessário no Google Colab)
import subprocess, sys

def instalar_se_necessario(pacote):
    try:
        __import__(pacote.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pacote, '-q'])

for pkg in ['scikit-posthocs', 'seaborn', 'jiwer']:
    instalar_se_necessario(pkg)


In [ ]:
# Importações principais
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scikit_posthocs as sp
from jiwer import cer as jiwer_cer, process_words
import warnings
import os

warnings.filterwarnings('ignore')
matplotlib.use('Agg')  # compatível com Colab e ambientes sem display

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"seaborn {sns.__version__}")
print(f"scipy   {stats.__version__ if hasattr(stats, '__version__') else 'OK'}")


In [ ]:
# ── Carregar dados ──────────────────────────────────────────────────────────
CSV_PATH = '../results/resultados_tts_tcc.csv'

# Suporte a execução no Google Colab (sem estrutura de pastas do repo)
if not os.path.exists(CSV_PATH):
    CSV_PATH = 'results/resultados_tts_tcc.csv'
if not os.path.exists(CSV_PATH):
    CSV_PATH = 'resultados_tts_tcc.csv'

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
print(f"Modelos: {df['Modelo'].unique().tolist()}")
print(f"Categorias ({df['Categoria'].nunique()}): {sorted(df['Categoria'].unique().tolist())}")


In [ ]:
# ── Primeiras linhas ─────────────────────────────────────────────────────────
display(df.head(6))


In [ ]:
# ── Tipos de dados e valores nulos ───────────────────────────────────────────
print("Tipos de dados:")
print(df.dtypes)
print()
print("Valores nulos por coluna:")
print(df.isna().sum())


---
## Seção 2 — Métricas Derivadas

Calculamos novas colunas a partir dos dados brutos para enriquecer a análise:

| Nova Coluna | Fórmula | Interpretação |
|---|---|---|
| `Throughput` | `len(Texto) / Latência` | Caracteres processados por segundo |
| `CER` | `jiwer.cer(texto, transcricao)` | Taxa de erro a nível de caractere |
| `WER_zero` | `WER == 0` | Flag de acerto perfeito |
| `Log_Latencia` | `log1p(Latência)` | Escala logarítmica para visualizações |

> **Nota:** Tratamento de edge cases: divisão por zero em Throughput (latência = 0), valores NaN em CER.


In [ ]:
# ── Throughput (caracteres/s) ─────────────────────────────────────────────────
df['Throughput'] = df.apply(
    lambda r: len(str(r['Texto'])) / r['Latência (s)'] if r['Latência (s)'] > 0 else np.nan,
    axis=1
)

# ── CER — Character Error Rate ────────────────────────────────────────────────
def calcular_cer(row):
    try:
        texto = str(row['Texto']).strip()
        transcricao = str(row['Transcrição Whisper']).strip()
        if not texto or not transcricao:
            return np.nan
        return jiwer_cer(texto, transcricao)
    except Exception:
        return np.nan

df['CER'] = df.apply(calcular_cer, axis=1)

# ── WER zero (acerto perfeito) ────────────────────────────────────────────────
df['WER_zero'] = df['WER'] == 0.0

# ── Log Latência (escala logarítmica) ────────────────────────────────────────
df['Log_Latencia'] = np.log1p(df['Latência (s)'])

print("Novas colunas adicionadas com sucesso!")
print(df[['Throughput', 'CER', 'WER_zero', 'Log_Latencia']].describe())


In [ ]:
# ── Verificação das novas métricas ───────────────────────────────────────────
print(f"Throughput — NaNs: {df['Throughput'].isna().sum()}")
print(f"CER        — NaNs: {df['CER'].isna().sum()}")
print(f"WER_zero   — True count: {df['WER_zero'].sum()} ({df['WER_zero'].mean()*100:.1f}% do total)")
print()
display(df[['Modelo', 'Texto', 'WER', 'CER', 'Throughput', 'WER_zero', 'Log_Latencia']].head(9))


---
## Seção 3 — Estatísticas Descritivas Expandidas

Tabela completa de estatísticas por modelo, incluindo intervalos de confiança de 95%
para WER e Latência, além de métricas de consistência (Coeficiente de Variação).


In [ ]:
def intervalo_confianca_95(serie):
    """Retorna (media, ci_lower, ci_upper) com IC 95% = média ± 1.96 * SEM."""
    s = serie.dropna()
    media = s.mean()
    sem = s.sem()
    return media, media - 1.96 * sem, media + 1.96 * sem

linhas = []
for modelo in ['Piper', 'Kokoro', 'XTTS']:
    sub = df[df['Modelo'] == modelo]
    wer = sub['WER']
    lat = sub['Latência (s)']

    wer_media, wer_ci_l, wer_ci_u = intervalo_confianca_95(wer)
    lat_media, lat_ci_l, lat_ci_u = intervalo_confianca_95(lat)

    wer_cv = (wer.std() / wer.mean() * 100) if wer.mean() > 0 else np.nan
    lat_cv = (lat.std() / lat.mean() * 100) if lat.mean() > 0 else np.nan

    linhas.append({
        'Modelo': modelo,
        'WER médio': round(wer_media, 4),
        'WER mediana': round(wer.median(), 4),
        'WER dp': round(wer.std(), 4),
        'WER mín': round(wer.min(), 4),
        'WER máx': round(wer.max(), 4),
        'WER CV (%)': round(wer_cv, 1),
        'WER IC95 inferior': round(wer_ci_l, 4),
        'WER IC95 superior': round(wer_ci_u, 4),
        'Latência média (s)': round(lat_media, 3),
        'Latência mediana (s)': round(lat.median(), 3),
        'Latência dp': round(lat.std(), 3),
        'Latência mín': round(lat.min(), 3),
        'Latência máx': round(lat.max(), 3),
        'Latência CV (%)': round(lat_cv, 1),
        'Latência IC95 inferior': round(lat_ci_l, 3),
        'Latência IC95 superior': round(lat_ci_u, 3),
        'Taxa WER zero (%)': round(sub['WER_zero'].mean() * 100, 1),
        'Throughput médio (char/s)': round(sub['Throughput'].mean(), 2),
        'CER médio': round(sub['CER'].mean(), 4),
    })

tabela_desc = pd.DataFrame(linhas).set_index('Modelo')
print("Estatísticas Descritivas Expandidas por Modelo:")
display(tabela_desc.T)


---
## Seção 4 — Testes Estatísticos

### Hipóteses Testadas
- **H₀:** Não há diferença estatisticamente significativa entre os modelos
- **H₁:** Há pelo menos um modelo com distribuição diferente

### Metodologia
- **Kruskal-Wallis** (não-paramétrico): comparação global entre os 3 modelos
- **Teste post-hoc de Dunn** (com correção de Bonferroni): comparação par-a-par
- **Correlação de Spearman**: relação entre NumPalavras e WER por modelo


In [ ]:
modelos = ['Piper', 'Kokoro', 'XTTS']
resultados_testes = []

# ── Kruskal-Wallis para WER ───────────────────────────────────────────────────
grupos_wer = [df[df['Modelo'] == m]['WER'].values for m in modelos]
H_wer, p_wer = stats.kruskal(*grupos_wer)

# ── Kruskal-Wallis para Latência ─────────────────────────────────────────────
grupos_lat = [df[df['Modelo'] == m]['Latência (s)'].values for m in modelos]
H_lat, p_lat = stats.kruskal(*grupos_lat)

print("=" * 60)
print("TESTE DE KRUSKAL-WALLIS")
print("=" * 60)
print(f"WER      → H = {H_wer:.4f}, p = {p_wer:.6f}  {'✅ Significativo (p<0,05)' if p_wer < 0.05 else '❌ Não significativo (p≥0,05)'}")
print(f"Latência → H = {H_lat:.4f}, p = {p_lat:.6f}  {'✅ Significativo (p<0,05)' if p_lat < 0.05 else '❌ Não significativo (p≥0,05)'}")

resultados_testes.append({'Teste': 'Kruskal-Wallis', 'Métrica': 'WER',      'Estatística (H)': round(H_wer, 4), 'p-value': round(p_wer, 6), 'Significativo (p<0.05)': p_wer < 0.05})
resultados_testes.append({'Teste': 'Kruskal-Wallis', 'Métrica': 'Latência', 'Estatística (H)': round(H_lat, 4), 'p-value': round(p_lat, 6), 'Significativo (p<0.05)': p_lat < 0.05})


In [ ]:
# ── Post-hoc de Dunn ─────────────────────────────────────────────────────────
print("=" * 60)
print("TESTE POST-HOC DE DUNN (correção de Bonferroni)")
print("=" * 60)

dunn_wer = sp.posthoc_dunn(df, val_col='WER',          group_col='Modelo', p_adjust='bonferroni')
dunn_lat = sp.posthoc_dunn(df, val_col='Latência (s)', group_col='Modelo', p_adjust='bonferroni')

print("\nDunn — WER:")
display(dunn_wer.round(4))

print("\nDunn — Latência:")
display(dunn_lat.round(4))

# Registrar pares
pares = [('Piper', 'Kokoro'), ('Kokoro', 'XTTS'), ('Piper', 'XTTS')]
for m1, m2 in pares:
    pw = dunn_wer.loc[m1, m2]
    pl = dunn_lat.loc[m1, m2]
    resultados_testes.append({'Teste': f'Dunn ({m1} vs {m2})', 'Métrica': 'WER',      'Estatística (H)': None, 'p-value': round(pw, 6), 'Significativo (p<0.05)': pw < 0.05})
    resultados_testes.append({'Teste': f'Dunn ({m1} vs {m2})', 'Métrica': 'Latência', 'Estatística (H)': None, 'p-value': round(pl, 6), 'Significativo (p<0.05)': pl < 0.05})


In [ ]:
# ── Correlação de Spearman: NumPalavras × WER ─────────────────────────────────
print("=" * 60)
print("CORRELAÇÃO DE SPEARMAN — NumPalavras × WER por modelo")
print("=" * 60)

for modelo in modelos:
    sub = df[df['Modelo'] == modelo]
    r, p = stats.spearmanr(sub['NumPalavras'], sub['WER'])
    forca = 'forte' if abs(r) >= 0.5 else ('moderada' if abs(r) >= 0.3 else 'fraca')
    print(f"{modelo:8s}: r = {r:+.4f}, p = {p:.4f}  → correlação {forca} {'(significativa)' if p < 0.05 else '(não significativa)'}")
    resultados_testes.append({'Teste': f'Spearman ({modelo})', 'Métrica': 'NumPalavras×WER', 'Estatística (H)': round(r, 4), 'p-value': round(p, 4), 'Significativo (p<0.05)': p < 0.05})

df_testes = pd.DataFrame(resultados_testes)


### Interpretação dos Resultados Estatísticos

Os testes de Kruskal-Wallis avaliam se existe diferença estatisticamente significativa entre as
distribuições de WER e Latência dos 3 modelos. O teste é não-paramétrico e não assume normalidade
dos dados — adequado para métricas como WER que costumam ter distribuição assimétrica.

O teste post-hoc de Dunn (com correção de Bonferroni para múltiplas comparações) identifica
**quais pares específicos** de modelos apresentam diferenças significativas.

A correlação de Spearman quantifica se frases mais longas (mais palavras) tendem a ter
maior ou menor WER — informação importante para entender as limitações de cada modelo.


---
## Seção 5 — Ranking por Categoria

Para cada uma das 14 categorias do corpus, identificamos qual modelo apresenta:
- **Menor WER** (melhor inteligibilidade)
- **Menor Latência** (mais rápido)


In [ ]:
# ── Pivot de WER e Latência por categoria ────────────────────────────────────
pivot_wer = df.pivot_table(
    index='Categoria', columns='Modelo', values='WER', aggfunc='mean'
).round(4)

pivot_lat = df.pivot_table(
    index='Categoria', columns='Modelo', values='Latência (s)', aggfunc='mean'
).round(3)

print("WER médio por categoria e modelo:")
display(pivot_wer)


In [ ]:
print("Latência média (s) por categoria e modelo:")
display(pivot_lat)


In [ ]:
# ── Vencedores por categoria ─────────────────────────────────────────────────
ranking = pd.DataFrame({'Categoria': pivot_wer.index})
ranking['Melhor WER (modelo)']       = pivot_wer.idxmin(axis=1).values
ranking['Melhor WER (valor)']        = pivot_wer.min(axis=1).round(4).values
ranking['Melhor Latência (modelo)']  = pivot_lat.idxmin(axis=1).values
ranking['Melhor Latência (valor s)'] = pivot_lat.min(axis=1).round(3).values
ranking = ranking.set_index('Categoria')

print("Vencedores por categoria:")
display(ranking)


In [ ]:
# ── Contagem de vitórias ──────────────────────────────────────────────────────
print("\nContagem de categorias vencidas:")
vit_wer = ranking['Melhor WER (modelo)'].value_counts()
vit_lat = ranking['Melhor Latência (modelo)'].value_counts()

cont = pd.DataFrame({
    'Vitórias em WER': vit_wer,
    'Vitórias em Latência': vit_lat
}).fillna(0).astype(int)
display(cont)


---
## Seção 6 — Análise de Erros

Decomposição detalhada dos tipos de erros de reconhecimento usando `jiwer.process_words()`:

| Tipo | Significado |
|---|---|
| **S — Substituições** | Palavra transcrita de forma errada |
| **D — Deleções** | Palavra do texto original não aparece na transcrição |
| **I — Inserções** | Palavra extra na transcrição não presente no original |

> Um WER alto causado principalmente por Substituições indica problema de pronúncia;
> Deleções indicam palavras suprimidas; Inserções indicam palavras fantasma adicionadas.


In [ ]:
# ── Contagem de tipos de erro por modelo ─────────────────────────────────────
def contar_erros_modelo(subdf):
    S_total = D_total = I_total = hits_total = 0
    for _, row in subdf.iterrows():
        try:
            ref = str(row['Texto']).strip()
            hyp = str(row['Transcrição Whisper']).strip()
            if not ref or not hyp:
                continue
            res = process_words(ref, hyp)
            S_total    += res.substitutions
            D_total    += res.deletions
            I_total    += res.insertions
            hits_total += res.hits
        except Exception:
            pass
    total_erros = S_total + D_total + I_total
    total_palavras = S_total + D_total + I_total + hits_total
    return {
        'Substituições (S)': S_total,
        'Deleções (D)': D_total,
        'Inserções (I)': I_total,
        'Total erros': total_erros,
        'S (%)': round(S_total / total_erros * 100, 1) if total_erros > 0 else 0,
        'D (%)': round(D_total / total_erros * 100, 1) if total_erros > 0 else 0,
        'I (%)': round(I_total / total_erros * 100, 1) if total_erros > 0 else 0,
        'Total palavras': total_palavras,
        'Acertos (%)': round(hits_total / total_palavras * 100, 1) if total_palavras > 0 else 0,
    }

analise_erros = {}
for modelo in modelos:
    analise_erros[modelo] = contar_erros_modelo(df[df['Modelo'] == modelo])

df_erros = pd.DataFrame(analise_erros).T
print("Análise de tipos de erro por modelo:")
display(df_erros)


In [ ]:
# ── Top 10 frases com maior WER por modelo ───────────────────────────────────
for modelo in modelos:
    print(f"\n{'='*70}")
    print(f"TOP 10 maiores WER — {modelo}")
    print('='*70)
    sub = df[df['Modelo'] == modelo].nlargest(10, 'WER')[
        ['FraseID', 'Texto', 'Transcrição Whisper', 'WER', 'CER']
    ].reset_index(drop=True)
    display(sub)


---
## Seção 7 — Gráficos Avançados

Geração e exportação de visualizações avançadas na pasta `results/`.

**Paleta de cores consistente por modelo:**
- 🔵 **Piper** = azul (`#2196F3`)
- 🟢 **Kokoro** = verde (`#4CAF50`)
- 🟠 **XTTS** = laranja (`#FF9800`)


In [ ]:
# ── Configurações globais de estilo ──────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

CORES = {'Piper': '#2196F3', 'Kokoro': '#4CAF50', 'XTTS': '#FF9800'}
ORDEM_MODELOS = ['Piper', 'Kokoro', 'XTTS']
DPI = 150

# Pasta de saída
RESULTS_DIR = '../results'
if not os.path.exists(RESULTS_DIR):
    RESULTS_DIR = 'results'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Gráficos serão salvos em: {os.path.abspath(RESULTS_DIR)}")


In [ ]:
# ── 1. Scatter WER vs Latência ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

for modelo in ORDEM_MODELOS:
    sub = df[df['Modelo'] == modelo]
    ax.scatter(
        sub['Latência (s)'], sub['WER'],
        c=CORES[modelo], label=modelo,
        s=sub['NumPalavras'] * 5,
        alpha=0.6, edgecolors='white', linewidths=0.5
    )

ax.set_xlabel('Latência (s)', fontsize=12)
ax.set_ylabel('WER', fontsize=12)
ax.set_title('WER vs Latência por Modelo\n(tamanho do ponto proporcional ao número de palavras)', fontsize=13)
ax.legend(title='Modelo', fontsize=11)

# Legenda de tamanho
for n in [5, 10, 20]:
    ax.scatter([], [], c='grey', alpha=0.5, s=n*5, label=f'{n} palavras')
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'scatter_wer_vs_latencia.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 2. Violin plot WER por modelo ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

sns.violinplot(
    data=df, x='Modelo', y='WER', order=ORDEM_MODELOS,
    palette=CORES, inner='box', ax=ax
)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('WER', fontsize=12)
ax.set_title('Distribuição de WER por Modelo (Violin Plot)', fontsize=13)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'violin_wer.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 3. Violin plot Latência por modelo ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

sns.violinplot(
    data=df, x='Modelo', y='Latência (s)', order=ORDEM_MODELOS,
    palette=CORES, inner='box', ax=ax
)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Latência (s)', fontsize=12)
ax.set_title('Distribuição de Latência por Modelo (Violin Plot)', fontsize=13)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'violin_latencia.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 4. Heatmap WER por modelo × categoria ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

heatmap_wer = df.pivot_table(
    index='Categoria', columns='Modelo', values='WER', aggfunc='mean'
)[ORDEM_MODELOS]

sns.heatmap(
    heatmap_wer, annot=True, fmt='.3f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'WER médio'}
)
ax.set_title('Heatmap: WER Médio por Modelo e Categoria', fontsize=13)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Categoria', fontsize=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'heatmap_wer_modelo_categoria.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 5. Heatmap Latência por modelo × categoria ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

heatmap_lat = df.pivot_table(
    index='Categoria', columns='Modelo', values='Latência (s)', aggfunc='mean'
)[ORDEM_MODELOS]

sns.heatmap(
    heatmap_lat, annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Latência média (s)'}
)
ax.set_title('Heatmap: Latência Média (s) por Modelo e Categoria', fontsize=13)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Categoria', fontsize=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'heatmap_latencia_modelo_categoria.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 6. Barras empilhadas — tipos de erro por modelo ──────────────────────────
erros_plot = df_erros[['Substituições (S)', 'Deleções (D)', 'Inserções (I)']].loc[ORDEM_MODELOS]

fig, ax = plt.subplots(figsize=(9, 6))

erros_plot.plot(
    kind='bar', stacked=True, ax=ax,
    color=['#e74c3c', '#3498db', '#2ecc71'],
    edgecolor='white', linewidth=0.5
)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Contagem de Erros', fontsize=12)
ax.set_title('Tipos de Erro de Reconhecimento por Modelo\n(Substituições, Deleções, Inserções)', fontsize=13)
ax.legend(title='Tipo de Erro', fontsize=10)
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'barras_tipos_erro.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 7. Radar chart comparativo ───────────────────────────────────────────────
eixos = ['WER\n(inv.)', 'Latência\n(inv.)', 'Taxa\nWER zero', 'Throughput', 'Consistência\n(1/CV)']
N = len(eixos)

metricas_raw = {}
for modelo in ORDEM_MODELOS:
    sub = df[df['Modelo'] == modelo]
    wer_med = sub['WER'].mean()
    lat_med = sub['Latência (s)'].mean()
    wer_zero_taxa = sub['WER_zero'].mean()
    throughput = sub['Throughput'].mean()
    wer_cv = sub['WER'].std() / sub['WER'].mean() if sub['WER'].mean() > 0 else np.nan
    consistencia = 1 / wer_cv if wer_cv and wer_cv > 0 else 0
    metricas_raw[modelo] = [wer_med, lat_med, wer_zero_taxa, throughput, consistencia]

# Normalizar 0-1 (invertendo WER e Latência: menor = melhor → maior = melhor)
vals = np.array([metricas_raw[m] for m in ORDEM_MODELOS])
min_vals = vals.min(axis=0)
max_vals = vals.max(axis=0)
rng = max_vals - min_vals
rng[rng == 0] = 1  # evitar divisão por zero

vals_norm = (vals - min_vals) / rng
# Inverter WER (índice 0) e Latência (índice 1): menor original → maior normalizado
vals_norm[:, 0] = 1 - vals_norm[:, 0]
vals_norm[:, 1] = 1 - vals_norm[:, 1]

# Plot
angulos = [n / float(N) * 2 * np.pi for n in range(N)]
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, modelo in enumerate(ORDEM_MODELOS):
    valores = vals_norm[i].tolist()
    valores += valores[:1]
    ax.plot(angulos, valores, 'o-', linewidth=2, color=CORES[modelo], label=modelo)
    ax.fill(angulos, valores, alpha=0.15, color=CORES[modelo])

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(eixos, size=11)
ax.set_yticklabels([])
ax.set_title('Radar Comparativo dos Modelos TTS\n(normalizado 0-1; maior = melhor)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'radar_comparativo.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


In [ ]:
# ── 8. Scatter WER vs NumPalavras com linha de tendência ─────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

for modelo in ORDEM_MODELOS:
    sub = df[df['Modelo'] == modelo].dropna(subset=['NumPalavras', 'WER'])
    ax.scatter(
        sub['NumPalavras'], sub['WER'],
        c=CORES[modelo], label=modelo, alpha=0.5, edgecolors='white', linewidths=0.3
    )
    # Linha de tendência (regressão linear)
    if len(sub) > 1:
        z = np.polyfit(sub['NumPalavras'], sub['WER'], 1)
        p_fit = np.poly1d(z)
        x_line = np.linspace(sub['NumPalavras'].min(), sub['NumPalavras'].max(), 100)
        ax.plot(x_line, p_fit(x_line), color=CORES[modelo], linewidth=2, linestyle='--')

ax.set_xlabel('Número de Palavras', fontsize=12)
ax.set_ylabel('WER', fontsize=12)
ax.set_title('WER vs Número de Palavras por Modelo\n(linha tracejada = tendência linear)', fontsize=13)
ax.legend(title='Modelo', fontsize=11)

plt.tight_layout()
caminho = os.path.join(RESULTS_DIR, 'scatter_wer_vs_numpalavras.png')
plt.savefig(caminho, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Salvo: {caminho}")


---
## Seção 8 — Exportação de Resultados Expandidos

Exportação de todos os DataFrames gerados neste notebook para arquivos CSV na pasta `results/`.


In [ ]:
# ── Resultados expandidos (DataFrame completo com novas colunas) ─────────────
caminho = os.path.join(RESULTS_DIR, 'resultados_expandidos.csv')
df.to_csv(caminho, index=False)
print(f"Salvo: {caminho}  ({df.shape[0]} linhas × {df.shape[1]} colunas)")

# ── Estatísticas descritivas ──────────────────────────────────────────────────
caminho = os.path.join(RESULTS_DIR, 'estatisticas_descritivas.csv')
tabela_desc.to_csv(caminho)
print(f"Salvo: {caminho}")

# ── Resultados dos testes estatísticos ───────────────────────────────────────
caminho = os.path.join(RESULTS_DIR, 'testes_estatisticos.csv')
df_testes.to_csv(caminho, index=False)
print(f"Salvo: {caminho}")

# ── Ranking por categoria ─────────────────────────────────────────────────────
caminho = os.path.join(RESULTS_DIR, 'ranking_por_categoria.csv')
ranking.to_csv(caminho)
print(f"Salvo: {caminho}")

# ── Análise de erros ──────────────────────────────────────────────────────────
caminho = os.path.join(RESULTS_DIR, 'analise_erros.csv')
df_erros.to_csv(caminho)
print(f"Salvo: {caminho}")

print("\n✅ Todos os arquivos exportados com sucesso!")


---
## Seção 9 — Conclusões Automáticas

Geração automática das conclusões baseadas nos dados calculados nas seções anteriores.


In [ ]:
# ── Coleta de valores para as conclusões ─────────────────────────────────────
melhor_wer_modelo = tabela_desc['WER médio'].idxmin()
melhor_wer_valor  = tabela_desc['WER médio'].min()

mais_rapido_modelo = tabela_desc['Latência média (s)'].idxmin()
mais_rapido_valor  = tabela_desc['Latência média (s)'].min()

mais_consistente_modelo = tabela_desc['WER CV (%)'].idxmin()
mais_consistente_cv     = tabela_desc['WER CV (%)'].min()

# Correlação de Spearman
correlacoes = {}
for modelo in ORDEM_MODELOS:
    sub = df[df['Modelo'] == modelo]
    r, p = stats.spearmanr(sub['NumPalavras'], sub['WER'])
    correlacoes[modelo] = (r, p)

# Modelo com correlação mais forte (maior |r|)
modelo_corr = max(correlacoes, key=lambda m: abs(correlacoes[m][0]))
r_corr, p_corr = correlacoes[modelo_corr]
forca_corr = 'forte' if abs(r_corr) >= 0.5 else ('moderada' if abs(r_corr) >= 0.3 else 'fraca/inexistente')

print("Conclusões geradas automaticamente:")
print(f"  Melhor WER: {melhor_wer_modelo} (WER = {melhor_wer_valor:.4f})")
print(f"  Mais rápido: {mais_rapido_modelo} (latência = {mais_rapido_valor:.3f} s)")
print(f"  Mais consistente: {mais_consistente_modelo} (CV = {mais_consistente_cv:.1f}%)")
print(f"  Kruskal-Wallis WER: H = {H_wer:.4f}, p = {p_wer:.6f}")
print(f"  Correlação NumPalavras×WER mais forte: {modelo_corr} (r = {r_corr:.4f}, {forca_corr})")


In [ ]:
# ── Template de conclusões preenchido automaticamente ─────────────────────────
from IPython.display import Markdown, display as ipy_display

sig_wer = p_wer < 0.05

conclusoes_md = f"""
## 📊 Conclusões da Análise Avançada

### Inteligibilidade (WER)
**O modelo com melhor inteligibilidade (menor WER) é {melhor_wer_modelo}** (WER médio = {melhor_wer_valor:.4f}).

### Velocidade (Latência)
**O modelo mais rápido é {mais_rapido_modelo}** (latência média = {mais_rapido_valor:.3f} s).

### Consistência (Coeficiente de Variação)
**O modelo mais consistente é {mais_consistente_modelo}** (CV do WER = {mais_consistente_cv:.1f}%).

### Significância Estatística
{'Há diferença estatisticamente significativa entre os modelos' if sig_wer else 'Não há diferença estatisticamente significativa entre os modelos'} (Kruskal-Wallis WER: H = {H_wer:.4f}, **p = {p_wer:.6f}**).

### Correlação Tamanho da Frase × WER
A correlação entre tamanho da frase e WER é **{forca_corr}** para o modelo **{modelo_corr}** (Spearman r = {r_corr:+.4f}, p = {p_corr:.4f}){' — estatisticamente significativa' if p_corr < 0.05 else ''}.

---
*Conclusões geradas automaticamente pelo notebook `notebooks/analise_avancada.ipynb`*
"""

ipy_display(Markdown(conclusoes_md))


---
## Referências

- Casanova, E. et al. (2024). *XTTS: A Massively Multilingual Zero-Shot Text-to-Speech Model*. Interspeech.
- Kim, J. et al. (2021). *VITS: Conditional Variational Autoencoder with Adversarial Learning for End-to-End TTS*. ICML.
- Li, Y. et al. (2023). *StyleTTS 2*. NeurIPS.
- Radford, A. et al. (2022). *Whisper: Robust Speech Recognition via Large-Scale Weak Supervision*. arXiv:2212.04356.
